<small><font color=gray>Notebook author: <a href="https://www.hse.ru/en/staff/sara/" target="_blank">Saraa Ali</a>  ©2026 onwards</font></small><hr style="margin:0;background-color:silver">

[**<font size=6>ML Hackathon 2026🦠Bacteria</font>**](https://www.kaggle.com/t/406899a3a00c45c5a9cef2d918bcaf51). [**Instructions**](https://colab.research.google.com/drive/1owkYjuRGkx050LQnM3b3yTzd0Dr2XbeV) for running Colabs.

In [1]:
from google.colab import drive; drive.mount('/content/drive')   # OK to enable, if your kaggle.json is stored in Google Drive

Mounted at /content/drive


In [2]:
!mkdir -p ~/.kaggle                               # .kaggle folder must contain kaggle.json for kaggle executable to properly authenticate you to Kaggle.com
!cp /content/drive/MyDrive/kaggle.json ~/.kaggle/kaggle.json >>log  # First, download kaggle.json from kaggle.com (in Account page) and place it in the root of mounted Google Drive
!cp kaggle.json ~/.kaggle/kaggle.json >> log      # Alternative location of kaggle.json (without a connection to Google Drive)
!chmod 600 ~/.kaggle/kaggle.json                  # give only the owner full read/write access to kaggle.json
!kaggle config set -n competition -v hse-ml-hackathon-2026 # set the competition context for the next few kaggle API calls. !kaggle config view - shows current settings
!kaggle competitions download >> log              # download competition dataset as a zip file
!unzip -o *.zip >> log                            # Kaggle dataset is copied as a single file and needs to be unzipped.
!kaggle competitions leaderboard --show           # print public leaderboard

cp: cannot stat 'kaggle.json': No such file or directory
- competition is now set to: hse-ml-hackathon-2026
100% 41.6M/41.6M [00:00<00:00, 218MB/s]
Using competition: hse-ml-hackathon-2026
Next Page Token = CfDJ8G82xNY93phHtA6xMwIf8xPpnuxAn9BhXGqi35467zFLuDOJnY6QnWFJXLTlJOhXKOvoi3aKSTiyYAvtupP_9Us
  teamId  teamName                submissionDate              score    
--------  ----------------------  --------------------------  -------  
16250206  E-chad gdp              2026-06-06 10:49:15.963000  0.98844  
16250271  AR-MA                   2026-06-06 10:54:06.223000  0.98496  
16250262  O_pus67                 2026-06-06 10:50:29.260000  0.98434  
16250227  F-ashion killas         2026-06-06 10:43:28.133000  0.98151  
16250202  BM-Bare_Minimum         2026-06-06 10:44:38.050000  0.98130  
16250380  AY                      2026-06-06 10:44:22.140000  0.98068  
16250203  AG                      2026-06-06 10:49:17.630000  0.97945  
16250210  X-XX                    2026-06-06 10:33:32

In [3]:
%%time
%%capture log
%reset -f
from IPython.core.interactiveshell import InteractiveShell as IS; IS.ast_node_interactivity = "all"
import numpy as np, pandas as pd, time, matplotlib.pyplot as plt, seaborn as sns, os
from sklearn.tree  import DecisionTreeClassifier
ToCSV = lambda df, fname: df.round(2).to_csv(f'{fname}.csv', index_label='id') # rounds values to 2 decimals

class Timer():
  def __init__(self, lim:'RunTimeLimit'=60): self.t0, self.lim, _ = time.time(), lim, print(f'started. You have {lim} sec. Good luck!')
  def ShowTime(self):
    msg = f'Runtime is {time.time()-self.t0:.0f} sec'
    print(f'[91m[1m' + msg + f' > {self.lim} sec limit!!![0m' if (time.time()-self.t0-1) > self.lim else msg)

np.set_printoptions(linewidth=100, precision=2, edgeitems=2, suppress=True)
pd.set_option('display.max_columns', 20, 'display.precision', 2, 'display.max_rows', 4)


CPU times: user 1.57 s, sys: 155 ms, total: 1.72 s
Wall time: 2.13 s


In [4]:
df = pd.read_csv('XY_Bacteria.csv'); df

,A0T0G0C10,A0T0G1C9,A0T0G2C8,A0T0G3C7,A0T0G4C6,A0T0G5C5,A0T0G6C4,A0T0G7C3,A0T0G8C2,A0T0G9C1,...,A8T0G1C1,A8T0G2C0,A8T1G0C1,A8T1G1C0,A8T2G0C0,A9T0G0C1,A9T0G1C0,A9T1G0C0,A10T0G0C0,y
0,10,32,41,39,77,122,55,81,58,31,...,38,88,80,20,36,31,32,32,10,NaN
1,10,31,52,165,225,221,150,143,48,31,...,111,125,143,159,69,45,32,71,10,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99998,10,31,66,107,142,155,142,107,66,31,...,93,66,93,93,66,31,31,31,10,4.0
99999,10,32,41,21,45,89,45,60,67,31,...,97,82,74,80,52,31,32,7,10,2.0


In [5]:
df.info()   # observe datatypes and any missing values

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Columns: 287 entries, A0T0G0C10 to y
dtypes: float64(1), int64(286)
memory usage: 219.0 MB


In [6]:
vX = df.query('y!=y').drop('y', axis=1)    # slice a test sample
tXY = df.query('y==y')                     # slice training sample
tX, tY = tXY.drop('y', axis=1), tXY.y.astype(int)      # split into training I/O
vX  # test inputs
tX  # train inputs
print(tY.tolist()[:50]) # train outputs

,A0T0G0C10,A0T0G1C9,A0T0G2C8,A0T0G3C7,A0T0G4C6,A0T0G5C5,A0T0G6C4,A0T0G7C3,A0T0G8C2,A0T0G9C1,...,A8T0G0C2,A8T0G1C1,A8T0G2C0,A8T1G0C1,A8T1G1C0,A8T2G0C0,A9T0G0C1,A9T0G1C0,A9T1G0C0,A10T0G0C0
0,10,32,41,39,77,122,55,81,58,31,...,27,38,88,80,20,36,31,32,32,10
1,10,31,52,165,225,221,150,143,48,31,...,76,111,125,143,159,69,45,32,71,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9998,10,31,66,107,142,156,143,107,66,31,...,420,296,303,421,302,306,31,31,31,10
9999,10,31,66,107,142,155,142,108,66,31,...,66,429,307,93,93,66,31,31,31,10


,A0T0G0C10,A0T0G1C9,A0T0G2C8,A0T0G3C7,A0T0G4C6,A0T0G5C5,A0T0G6C4,A0T0G7C3,A0T0G8C2,A0T0G9C1,...,A8T0G0C2,A8T0G1C1,A8T0G2C0,A8T1G0C1,A8T1G1C0,A8T2G0C0,A9T0G0C1,A9T0G1C0,A9T1G0C0,A10T0G0C0
10000,10,7,48,74,119,110,127,86,66,7,...,223,319,233,361,439,368,179,174,199,10
10001,10,31,66,107,142,156,142,108,66,31,...,66,539,302,93,424,434,31,313,31,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99998,10,31,66,107,142,155,142,107,66,31,...,66,93,66,93,93,66,31,31,31,10
99999,10,32,41,21,45,89,45,60,67,31,...,52,97,82,74,80,52,31,32,7,10


[3, 3, 4, 0, 4, 3, 4, 4, 3, 3, 2, 2, 4, 2, 2, 4, 4, 2, 2, 0, 2, 3, 3, 3, 4, 3, 3, 4, 3, 3, 3, 4, 4, 2, 3, 3, 2, 0, 1, 4, 4, 2, 3, 4, 3, 0, 3, 2, 2, 0]


In [7]:
pd.set_option('display.max_rows', 10)
# tY.value_counts()
df['y'].value_counts(normalize=True)

,proportion
y,
4.0,0.30
2.0,0.30
3.0,0.30
1.0,0.05
0.0,0.05


In [68]:
tmr = Timer()

started. You have 60 sec. Good luck!


<hr color=green size=40>

<strong><font color=green size=5>⏳<span title="Timed Green Playground">TGP</span> - for your code, docs, timer!</font></strong>

<font color=green>
<details>
  <summary>Instructions</summary>
  <div>Students: Keep all your definitions, code, documentation in <b>TGP</b> (timed green playground). Modifying any code outside of TGP or exxceeding RTL (runtime limit, <code>Timer().lim</code>) incurs penalties - see <a href=https://drive.google.com/drive/folders/10_OAoFTdUg71Z0Op_ebxIxcNfQWByakT?usp=drive_link>Grading Rubric for Kaggle Colabs</a> Google Sheet.
</div> </details>
</font>

<font color=green><h3><b>$\alpha$. Split observations into train and test sets</b><h3>

In [69]:
%%time
tX_in = tX.apply(pd.to_numeric, errors='coerce')
tY_in = tY.astype(int)
vX_in = vX.apply(pd.to_numeric, errors='coerce')


CPU times: user 50 ms, sys: 15 ms, total: 64.9 ms
Wall time: 64.6 ms


<font color=green><h3><b>$\beta$. Build and fit a model</b><h3>

In [70]:
%%time
from sklearn.ensemble import HistGradientBoostingClassifier
m = HistGradientBoostingClassifier(
    max_iter=80,
    max_depth=12,
    learning_rate=0.12,
    min_samples_leaf=5,
    l2_regularization=0.0,
    class_weight='balanced',
    random_state=42
)

m.fit(tX_in, tY_in)


CPU times: user 58.7 s, sys: 141 ms, total: 58.9 s
Wall time: 59.2 s


HistGradientBoostingClassifier(class_weight='balanced', learning_rate=0.12,
                               max_depth=12, max_iter=80, min_samples_leaf=5,
                               random_state=42)

<font color=green><h3><b>$\gamma$. Make predictions</b><h3>

In [71]:
pY = pd.DataFrame(m.predict(vX_in).astype(int), index=vX.index + 1, columns=['y'])
ToCSV(pY, 'submission')
print('Observed  y distribution: ', list((tXY.y.value_counts()/len(tXY)).round(3)))
print('Predicted y distribution: ', list((pY.round(0).astype(int).value_counts()/len(pY)).round(3)))

Observed  y distribution:  [0.303, 0.302, 0.296, 0.05, 0.048]
Predicted y distribution:  [0.304, 0.303, 0.296, 0.049, 0.048]


<font color=green><h3><b>$\delta$. Idea Documentation</b></h3>
<details>
  <summary>Instructions</summary>
  <div>


1. **Audience**. Your peers who will learn from your Colab and ideas therein.
1. **Importance**. The ML ideas are not entirely random, but are based on prior experience and systematized/organized experiments. We'd like students to share and learn from idea generation to idea experimentation process done in our class using tools learned thus far.
1. **Format**. Keep it concise/precise in consistent font/presentation. Include numbers/IDs to your References, such as [1] or [[Géron22]](https://scholar.google.com/scholar?cluster=498861685923226475), where these are defined in your References section below. This helps link your ideas/experiments to external ideas.
1. **Reproducibility**. Your description should contain reasonable details needed for reproducibility, i.e. describe the state of your modeling pipeline before the change is made, what is changed and how the idea was discovered, and what improvement it resulted in. Thus, peers can try this idea with an expectation of the value it brings. See examples below.
1. **Bonus** points for the exceptional/exemplary/educational documentation (see grading rubric).
****
1. **TODO**: Describe the key idea in your work in the following format (similar to a "micro publication"):
  1. **Title**. Give each idea a descriptive name (i.e. a micro abstract).
    1. Ex(ample). <i>"Thresholding carat feature outliers improves MAE by 3% on public LB"</i>
  1. **Idea Discovery**. What led you to this idea? Was it some [EDA](https://en.wikipedia.org/wiki/Exploratory_data_analysis), familiarity with this dataset or some of the features?
    1. Ex. <i>"We plotted all univariate distributions of variables and discovered that diamond carat had unreasonable (but rare) values below and above [0,10] interval, when plotted carat's histogram in the train and test sets, which contained 10 and 3 such outliers respectively. We decided to use 10 as a reasonable threshold because it is 99th percentile of carat values in the 20K baseline sample. See our histogram plot below [plot here]. "</i>
  1. **Finding's Importance**. Describe why you think the idea was important to proceed with.
    1. Ex. <i>"We use a linear model, the slope of which is sensitive to outliers on the periphery of the feature space domain. The fitted hyperplane slopes in the direction of the extreme training feature values thereby mapping a non-existent relation between carat size and diamond price, which is not expected to repeat in the test set. "</i>
  1. **Experiment Setup**.
  How did you set up experiments to test your idea? What resources were helpful? What metric did you select, why and what values did you observe?
    1. Ex. <i>"To alleviate the impact of the outlying feature values, we need to either remove observations with extreme values, or somehow cap them (to stay within the distribution of the other carat values) or use a model insensitive to outliers (such as robust regression). We learned 3 suitable methods for treating outliers in [ref]: ... [It'd be great to briefly describe each method] We tried each one on a Baseline model, while keeping the competition-required [MAE](https://en.wikipedia.org/wiki/Mean_absolute_error) metric. We tested each method locally on the seeded 50/50 split of the 20K training set sampled in baseline Colab."</i>
  1. **Results**. What was the result or metric improvement from implementing the experiment locally and/or on public LB?
    1. Ex. <i>"Baseline MAE was 539.1257546465 in public LB and 530 in local default experiment with 50/50 train-test split. When applied on the same-seed split, Methods 1,2,and 3 showed 1%, 2%, and 5% improvement on the test set. When uploaded to public LB, Method 3 showed a 3% improvement. So, we decided to keep method 3."</i>

</div> </details>
</font>


<font color=green><h4><b>Task 1. Preprocessing Ideas</b></h4>
<details>
  <summary>Instructions</summary>
  <div>Explain a <b>key idea</b> that helped in <b>preprocessing pipeline</b>. This may be about some feature engineering, tricky subsampling, clustering, dimension reduction, etc. Use the format in TODO specified above. Remember to provide citation references for the peers to read more into your work.
</div> </details>
</font>

1. **Title**: use class_weight='balanced' to handle class imbalance (5% and 30%) improves score
1. **Idea Discovery**: when we looked at the class distribution in the training data, we noticed a strong imbalance. classes 0 and 1 make up only 5% each, while classes 2, 3, and 4 have 30% each. since the evaluation metric weights all classes equally, a model that ignores the rare classes would score much lower.
1. **Finding's Importance**: the baseline model almost never predicts classes 0 and 1, but the evaluation metric treats all classes equally. to fix this, we used class_weight='balanced', which makes the model pay more attention to rare classes by increasing their weight during training.
1. **Experiment Setup**: we tested 3 approaches : no balancing, class_weight='balanced', manual oversampling of minority classes. we used HistGradientBoostingClassifier with max_iter=80, max_depth=12 as the base model.
1. **Results**: no balancing gave about 0.942 score, class_weight='balanced' 0.977 and oversampling about 0.971. we chose the second approach.

<font color=green><h4><b>Task 2. Modeling Ideas</b></h4>
<details>
  <summary>Instructions</summary>
  <div>Explain a <b>key idea</b> that helped with <b>model selection</b> in the format specified above. This may include tuning model parameters (perhaps a grid search with specific parameter range) or some other experiments, search/choice of the suitable model, experiments with postprocessing of model predictions, etc. Use the format in TODO specified above. Remember to provide citation references for the peers to read more into your work.
</div> </details>
</font>

1. **Title**: replacing DecisionTree with HistGradientBoosting improves Macro-F1 from 0.641 to 0.977
1. **Idea Discovery**: the baseline uses a single decision tree with depth 5, which is too simple for 286 features. we switched to HistGradientBoostingClassifier, which builds many trees one by one, where each tree fixes the mistakes of the previous one. it trains fast and works well with this kind of data.
1. **Finding's Importance**: a depth-5 tree can only split the data into 32 groups. it is too few for 286 features. using 80 deeper trees with depth 12 allows the model to find complex patterns between nucleotide counts that one small tree would miss.
1. **Experiment Setup**: we tried max_iter (80–150), max_depth (5–12), learning_rate (0.05–0.15) and min_samples_leaf (2–20), evaluating each with score. also tested VotingClassifier but it gave poor results. all models used class_weight='balanced'.
1. **Results**: the baseline DecisionTree scored 0.641 on the leaderboard. switching to HGB at depth 6 already jumped to 0.966. depth 8 reached 0.976, and depth 12 with iter=80 matched score 0.977. ExtraTrees hit 0.970 on score was faster but weaker. VotingClassifier gave 0.973, no improvement over HGB.

<font color=green><h3><b>$\epsilon$. References</b></h3>
<details>
  <summary>Instructions</summary>
  <div>

1. Cite your sources to help your peers learn from these (and to avoid plagiarism).
1. Course textbooks should be cited when they informed your work.
1. Use Google Scholar to draw [APA](https://en.wikipedia.org/wiki/American_Psychological_Association) citation format for books and publications.
1. Cite [StackOverflow](https://stackoverflow.com/), YouTube videos, package docs, open-access textbooks/publications and other meaningful internet resources that you used.
1. We may reward exceptional and meaningful citations. Do not rely only on package manual pages.

</div> </details>
</font>


1. https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.HistGradientBoostingClassifier.html
1. https://scikit-learn.org/stable/modules/generated/sklearn.utils.class_weight.compute_class_weight.html
1. https://scikit-learn.org/stable/modules/ensemble.html#histogram-based-gradient-boosting

<font color=green><h4><b>$\epsilon$. LLM Documentation if used</b></h4></font>

LLM tool used: Antigravity IDE assistant: https://claude.ai

how it was used: the LLM helped with preprocessing, submission formatting and testing different parameter combinations. all final parameter values were selected based on our own experiments.

prompts, which were used: "how to convert all columns to numeric in pandas?" and"what does class_weight='balanced' do in sklearn?"

<hr color=green size=40><strong><font color=green size=5>⌛End of TGP. Do not exceed RTL! Do not write code outside TGP.</font></strong>
<!--<hr color=green size=40>-->

In [72]:
tmr.ShowTime()    # measure Colab's runtime. Do not remove. Keep as the last cell in your notebook.

Runtime is 60 sec


In [73]:
!kaggle competitions submit -f submission.csv -m "solution" -c hse-ml-hackathon-2026


100% 67.3k/67.3k [00:00<00:00, 151kB/s]
Successfully submitted to HSE ML Hackathon 2026

<details>
  <summary><font size=5><b>Starter Ideas</b></font></summary>
  <div>

Use methods such as `LogisticRegression`, `RidgeClassifier`, `KNeighborsClassifier`, `LinearSVC`, `RandomForestClassifier`, `ExtraTreesClassifier`, `StandardScaler`, and `PCA` inside `Pipeline`.

</div> </details>
